# Decoder-Only Transformer

A minimal GPT-style decoder trained on two sentences. Everything is tiny on purpose:
`vocab_size = 5`, `d_model = 4`, one attention head, one decoder block.


In [163]:
import math
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [164]:
text = """
Ryan was the captain of his school football team. 
Every Saturday afternoon his team played against another school in the local league. 
Ryan was a fast striker who scored many goals, while his friend Daniel was the goalkeeper who saved difficult shots. 
During one important match, the score was tied at one goal each with only five minutes remaining. 
Daniel made an excellent save and quickly passed the ball to Ryan. 
Ryan dribbled past two defenders, kicked the ball into the top corner of the goal, and the crowd cheered loudly. 
After the match, the coach congratulated the players for their teamwork and reminded them that passing the ball was more important than playing alone.
The team celebrated the victory together and started preparing for their next game.
"""

tokens = re.findall(r"\w+|[.,!?]", text.lower())

special_tokens = ["<PAD>", "<EOS>"]

vocab = sorted(set(tokens))


token_to_id = {}

# Add special tokens first
for token in special_tokens:
    token_to_id[token] = len(token_to_id)

    
# Add vocabulary
for token in vocab:
    token_to_id[token] = len(token_to_id)

id_to_token = {v: k for k, v in token_to_id.items()}

vocab_size = len(token_to_id)
d_model = 16
max_len = 32

In [165]:
# =====================================================
# Training Sentences
#
# The first <EOS> separates the prompt from the answer.
# The trailing <EOS> is the real end of the sequence -- without
# it, "awesome" is never seen as an input, so the model has no
# idea what follows it and generation can't learn to stop.
# =====================================================

# Split into sentences on . ! ?
sentences = [
    s.strip()
    for s in re.split(r"[.!?]+", text)
    if s.strip()
]

# Tokenize each sentence the SAME way the vocab was built
# (re.findall separates punctuation, so "goals," -> ["goals", ","]).
sentences = [
    re.findall(r"\w+|[.,!?]", s.lower()) + ["<EOS>"]
    for s in sentences
]

max_len = max(len(s) for s in sentences)

dataset = [
    s + ["<PAD>"] * (max_len - len(s))
    for s in sentences
]

print(id_to_token)

print(dataset)

{0: '<PAD>', 1: '<EOS>', 2: ',', 3: '.', 4: 'a', 5: 'after', 6: 'afternoon', 7: 'against', 8: 'alone', 9: 'an', 10: 'and', 11: 'another', 12: 'at', 13: 'ball', 14: 'captain', 15: 'celebrated', 16: 'cheered', 17: 'coach', 18: 'congratulated', 19: 'corner', 20: 'crowd', 21: 'daniel', 22: 'defenders', 23: 'difficult', 24: 'dribbled', 25: 'during', 26: 'each', 27: 'every', 28: 'excellent', 29: 'fast', 30: 'five', 31: 'football', 32: 'for', 33: 'friend', 34: 'game', 35: 'goal', 36: 'goalkeeper', 37: 'goals', 38: 'his', 39: 'important', 40: 'in', 41: 'into', 42: 'kicked', 43: 'league', 44: 'local', 45: 'loudly', 46: 'made', 47: 'many', 48: 'match', 49: 'minutes', 50: 'more', 51: 'next', 52: 'of', 53: 'one', 54: 'only', 55: 'passed', 56: 'passing', 57: 'past', 58: 'played', 59: 'players', 60: 'playing', 61: 'preparing', 62: 'quickly', 63: 'remaining', 64: 'reminded', 65: 'ryan', 66: 'saturday', 67: 'save', 68: 'saved', 69: 'school', 70: 'score', 71: 'scored', 72: 'shots', 73: 'started', 74: '

In [166]:
# =====================================================
# Build Training Data
# Input  = sentence[:-1]
# Target = sentence[1:]
# =====================================================
X = []
Y = []

for sentence in dataset:
    ids = [token_to_id[word if word.startswith("<") else word.lower()] for word in sentence]
    X.append(ids[:-1])
    Y.append(ids[1:])

X = torch.tensor(X)
Y = torch.tensor(Y)

print("Input")
print(X)

print("\nTarget")
print(Y)

Input
tensor([[65, 88, 79, 14, 52, 38, 69, 31, 75,  1,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0],
        [27, 66,  6, 38, 75, 58,  7, 11, 69, 40, 79, 44, 43,  1,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0],
        [65, 88,  4, 29, 74, 90, 71, 47, 37,  2, 89, 38, 33, 21, 88, 79, 36, 90,
         68, 23, 72,  1,  0,  0,  0],
        [25, 53, 39, 48,  2, 79, 70, 88, 82, 12, 53, 35, 26, 91, 54, 30, 49, 63,
          1,  0,  0,  0,  0,  0,  0],
        [21, 46,  9, 28, 67, 10, 62, 55, 79, 13, 83, 65,  1,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0],
        [65, 24, 57, 86, 22,  2, 42, 79, 13, 41, 79, 85, 19, 52, 79, 35,  2, 10,
         79, 20, 16, 45,  1,  0,  0],
        [ 5, 79, 48,  2, 79, 17, 18, 79, 59, 32, 80, 76, 10, 64, 81, 78, 56, 79,
         13, 88, 50, 39, 77, 60,  8],
        [79, 75, 15, 79, 87, 84, 10, 73, 61, 32, 80, 51, 34,  1,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0]])

Target
tensor([[88, 79, 14, 52, 38, 69, 

In [167]:
# =====================================================
# Positional Encoding
# =====================================================
class PositionEncoding(nn.Module):

    def __init__(self, d_model, max_len):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(max_len).unsqueeze(1)

        embedding_index = torch.arange(0, d_model, 2)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

       
        self.register_buffer("pe", pe)

    def forward(self, word_embeddings):
        seq_len = word_embeddings.size(-2)
        return word_embeddings + self.pe[:seq_len]


In [168]:
# =====================================================
# Single Head Masked Attention
# =====================================================
class Attention(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(d_model)

        seq_len = x.size(1)

        mask = torch.triu(
            torch.ones(seq_len, seq_len),
            diagonal=1
        ).bool()

        scores = scores.masked_fill(mask, -1e9)

        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)

        return output

In [169]:
# =====================================================
# Decoder Block
# =====================================================
class DecoderBlock(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.attention = Attention(d_model)

        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 8),
            nn.ReLU(),
            nn.Linear(8, d_model),
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        attn = self.attention(x)

        x = self.norm1(x + attn)

        ff = self.ff(x)

        x = self.norm2(x + ff)

        return x

In [170]:

# =====================================================
# Decoder Only Transformer
# =====================================================
class DecoderOnlyTransformer(nn.Module):

    def __init__(self, vocab_size, d_model, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position = PositionEncoding(d_model, max_len)
        self.decoder = DecoderBlock(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.position(x)
        x = self.decoder(x)
        logits = self.fc(x)
        return logits



In [171]:

# =====================================================
# Create Model
# =====================================================
model = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    d_model=d_model,
    max_len=max_len,
)

criterion = nn.CrossEntropyLoss(ignore_index=token_to_id["<PAD>"])

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
)


In [172]:
# =====================================================
# Training
# =====================================================
epochs = 1000

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    loss = criterion(
        logits.reshape(-1, vocab_size),
        Y.reshape(-1),
    )

    loss.backward()

    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}  Loss = {loss.item():.4f}")


Epoch    0  Loss = 4.6743
Epoch  100  Loss = 0.0407
Epoch  200  Loss = 0.0296
Epoch  300  Loss = 0.0273
Epoch  400  Loss = 0.0263
Epoch  500  Loss = 0.0258
Epoch  600  Loss = 0.0257
Epoch  700  Loss = 0.0254
Epoch  800  Loss = 0.0253
Epoch  900  Loss = 0.0252


In [173]:
# =====================================================
# Predictions
# =====================================================
print("\nPredictions")

model.eval()

with torch.no_grad():

    logits = model(X)

    predictions = logits.argmax(dim=-1)

print(predictions)

print("\nDecoded Predictions")

for sentence in predictions:

    words = [id_to_token[token.item()] for token in sentence]

    print(words)


Predictions
tensor([[88,  4, 14, 52, 38, 69, 31, 75,  1, 22, 42, 18, 18, 42, 42, 55, 62, 62,
         62, 55, 62, 62, 62, 62, 62],
        [66,  6, 38, 75, 58,  7, 11, 69, 40, 79, 44, 43,  1, 79, 42, 79, 62, 62,
         62, 62, 62, 62, 62, 62, 62],
        [88,  4, 29, 74, 90, 71, 47, 37,  2, 89, 38, 33, 21, 88, 79, 36, 90, 68,
         23, 72,  1, 79, 28, 62, 62],
        [53, 39, 48,  2, 79, 70, 88, 82, 12, 53, 35, 26, 91, 54, 30, 49, 63,  1,
          1, 18, 62, 62, 62, 62, 62],
        [46,  9, 28, 67, 10, 62, 55, 79, 13, 83, 65,  1, 59, 55, 55, 62, 62, 62,
         62, 62, 62, 62, 62, 62, 62],
        [88, 57, 86, 22,  2, 42, 79, 13, 41, 79, 85, 19, 52, 79, 35,  2, 10, 79,
         20, 16, 45,  1,  1, 79, 79],
        [79, 48,  2, 79, 17, 18, 79, 59, 32, 80, 76, 10, 64, 81, 78, 56, 79, 13,
         88, 50, 39, 77, 60,  8,  1],
        [75, 15, 79, 87, 84, 10, 73, 61, 32, 80, 51, 34,  1, 53, 79, 79, 79, 62,
         62, 62, 62, 62, 62, 62, 62]])

Decoded Predictions
['was', 'a', 

In [176]:
# =====================================================
# Autoregressive Generation
#
# <EOS> here is the prompt/response separator, not the end
# of the sequence -- the answer comes after it. So the first
# <EOS> is passed over and we stop on the second one.
# =====================================================
print("\nGeneration")

prompt = ["daniel"]

generated = prompt.copy()

seen_eos = False

for _ in range(max_len - len(generated)):

    ids = [token_to_id[word] for word in generated]

    x = torch.tensor([ids])

    with torch.no_grad():
        logits = model(x)

    next_token = logits[0, -1].argmax().item()

    next_word = id_to_token[next_token]

    generated.append(next_word)

    if next_word == "<EOS>":

        if seen_eos:
            break

        seen_eos = True

output = " ".join(word for word in generated if word != "<EOS>")
print("Prompt :", prompt)
print("Output :", output)


Generation
Prompt : ['daniel']
Output : daniel made an excellent save and quickly passed the ball to ryan players with only
